# ASG Airlines — Data Profiling

### Objective

The goal of this notebook is to understand the structure and quality of the raw airline dataset before building the ETL pipeline.

**Tasks**
- Load all Excel sheets
- Inspect schema
- Identify missing values
- Detect duplicates
- Understand relationships between tables
- Generate a data quality summary

In [9]:
Installing Libraries


SyntaxError: invalid syntax (2748646108.py, line 1)

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [ ]:
Loading Excel Sheet 

In [ ]:
# Project root
DATA_PATH = Path("../data/raw/UseCase - Airlines.xlsx")

# Read workbook
excel_file = pd.ExcelFile(DATA_PATH)

# List all sheet names
excel_file.sheet_names

['flights', 'payments', 'bookings', 'passengers']

In [ ]:
# Dictionary containing every dataframe
dfs = {}

for sheet in excel_file.sheet_names:
    dfs[sheet] = pd.read_excel(excel_file, sheet_name=sheet)

print(f"Total Sheets : {len(dfs)}")
print(dfs.keys())

Total Sheets : 4
dict_keys(['flights', 'payments', 'bookings', 'passengers'])


In [ ]:
for name, df in dfs.items():
    print("="*60)
    print(f"Sheet : {name}")
    print("="*60)

    display(df.head())

Sheet : flights


,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


Sheet : payments


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


Sheet : bookings


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


Sheet : passengers


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


## Data Quality Assessment

We will evaluate the dataset on six dimensions:

1. Missing Values
2. Duplicate Records
3. Primary Key Integrity
4. Foreign Key Integrity
5. Invalid Datatypes
6. Business Rule Validation

In [14]:
Data Quality Report 

SyntaxError: invalid syntax (1526385566.py, line 1)

In [ ]:


summary = []

for name, df in dfs.items():

    summary.append({
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isna().sum().sum(),
        "Duplicate Rows": df.duplicated().sum()
    })

quality_report = pd.DataFrame(summary)
quality_report

,Table,Rows,Columns,Missing Values,Duplicate Rows
0,flights,1020,7,41,15
1,payments,1000,4,48,0
2,bookings,1000,9,45,0
3,passengers,1039,9,10,0


Primary Key Integrity check 

In [ ]:
primary_keys = {
    "flights": "flight_id",
    "passengers": "passenger_id",
    "bookings": "booking_id",
    "payments": "payment_id"
}

pk_report = []

for table, pk in primary_keys.items():

    df = dfs[table]

    pk_report.append({
        "Table": table,
        "Primary Key": pk,
        "Null IDs": df[pk].isna().sum(),
        "Duplicate IDs": df[pk].duplicated().sum()
    })

pd.DataFrame(pk_report)

,Table,Primary Key,Null IDs,Duplicate IDs
0,flights,flight_id,0,16
1,passengers,passenger_id,0,39
2,bookings,booking_id,0,0
3,payments,payment_id,0,0


Looking into column wise missing  values as they can be  spread across multiple columns.

In [ ]:


for name, df in dfs.items():

    print("\n" + "="*60)
    print(f"{name.upper()} - Missing Values")
    print("="*60)

    missing = df.isna().sum()

    missing = missing[missing > 0].sort_values(ascending=False)

    display(missing.to_frame("Missing Count"))


FLIGHTS - Missing Values


,Missing Count
airline,41



PAYMENTS - Missing Values


,Missing Count
amount,48



BOOKINGS - Missing Values


,Missing Count
status,45



PASSENGERS - Missing Values


,Missing Count
last_name,10


Investigating Duplicates

Flights has duplicate rows and duplicate IDs

Passengers has 39 duplicate IDs but 0 duplicate rows

That means passenger records are not identical—the same passenger ID appears with different data.

Flight Duplicate Investigation

In [15]:
flight_duplicates = (
    dfs["flights"]
    .loc[dfs["flights"]["flight_id"].duplicated(keep=False)]
    .sort_values("flight_id")
)

flight_duplicates

,flight_id,airline,source,destination,departure_time,arrival_time,duration
253,6F250,UNKNOWN,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00
270,6F250,UNKNOWN,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00
550,AI020,NaN,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,02:28:00
549,AI020,NaN,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,02:28:00
141,AI031,Air India,DEL,MAA,2026-04-20 13:05:41.701,2026-04-20 16:11:41.701,03:06:00
...,...,...,...,...,...,...,...
813,UK160,Vistara,BLR,BOM,2026-04-18 06:42:41.703,2026-04-18 10:58:41.703,04:16:00
503,UK163,Vistara,DEL,HYD,2026-04-19 07:22:41.703,2026-04-19 09:04:41.703,01:42:00
502,UK163,Vistara,DEL,HYD,2026-04-19 07:22:41.703,2026-04-19 09:04:41.703,01:42:00
914,UK180,Vistara,CCU,MAA,2026-04-17 21:18:41.703,2026-04-17 23:29:41.703,02:11:00


Passenger Duplicate Investigation


In [16]:
passenger_duplicates = (
    dfs["passengers"]
    .loc[dfs["passengers"]["passenger_id"].duplicated(keep=False)]
    .sort_values("passenger_id")
)

passenger_duplicates

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
34,P1034,Ishaan,Iyer,62,M,ishaan.iyer@outlook.com,+91-8679984033,122362316658,1964-08-14
35,P1034,Ishaan,Iyer,62,M,ishaan.iyer@gmail.com,+91-7994310083,669096705466,1964-09-24
103,P1102,Myra,Kulkarni,1,F,myra.kulkarni@rediffmail.com,+91-8128636900,606853615305,2025-02-27
104,P1102,Myra,Kulkarni,1,F,myra.kulkarni@outlook.com,+91-7247746438,47277901043,2025-03-18
106,P1104,Vivaan,Verma,88,F,vivaan.verma@gmail.com,+91-7871531100,246095396218,1938-06-03
...,...,...,...,...,...,...,...,...,...
1028,P1991,Aarav,Shhharma,4,F,aarav.shhharma@rediffmail.com,+91-9196057919,645345472239,2022-11-17
1031,P1994,Diya,Kulkarni,89,F,diya.kulkarni1@hotmail.com,+91-7409498553,876230391114,1937-03-18
1032,P1994,DiyaKulkarni,NaN,89,F,diyakulkarni.user@gmail.com,+91-9694250229,522823052480,1937-09-17
1037,P1999,Ananya,Pillai,14,F,ananya.pillai2@yahoo.com,+91-9733969290,246320374767,2012-12-15


## Profiling Summary

### Dataset Overview

| Table | Rows | Missing | Duplicate PK |
|--------|------|----------|--------------|
| Flights | 1020 | 41 | 16 |
| Passengers | 1039 | 10 | 39 |
| Bookings | 1000 | 45 | 0 |
| Payments | 1000 | 48 | 0 |

### Key Findings

- Flights contain duplicate flight IDs and missing airline values.
- Passenger master contains duplicate passenger IDs.
- Booking status has missing values.
- Payment amounts contain missing values and require business validation.
- The dataset contains PII (Aadhaar, Passport, Email, Phone) that must be masked before reporting.

In [17]:
# Compare duplicate passenger records

passenger_duplicates.sort_values(["passenger_id"]).head(20)

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
34,P1034,Ishaan,Iyer,62,M,ishaan.iyer@outlook.com,+91-8679984033,122362316658,1964-08-14
35,P1034,Ishaan,Iyer,62,M,ishaan.iyer@gmail.com,+91-7994310083,669096705466,1964-09-24
103,P1102,Myra,Kulkarni,1,F,myra.kulkarni@rediffmail.com,+91-8128636900,606853615305,2025-02-27
104,P1102,Myra,Kulkarni,1,F,myra.kulkarni@outlook.com,+91-7247746438,47277901043,2025-03-18
106,P1104,Vivaan,Verma,88,F,vivaan.verma@gmail.com,+91-7871531100,246095396218,1938-06-03
...,...,...,...,...,...,...,...,...,...
224,P1216,Riya,Sharma,56,F,riya.sharma@outlook.com,+91-7367942460,154100676267,1970-07-12
225,P1216,Riya,Sharma,56,F,riya.sharma@hotmail.com,+91-9208908082,793271040541,1970-02-11
245,P1236,Myra,Sharma,20,F,myra.sharma@yahoo.com,+91-9821413281,584282417248,2006-03-22
246,P1236,MyraSharma,NaN,20,F,myrasharma.user@outlook.com,+91-6338642889,534655494717,2006-01-28


## Master Data Resolution Strategy

Duplicate passenger IDs contain conflicting personal information.
Since no source system priority is provided, records will be resolved using the following rules:

1. Keep the record with the highest completeness score.
2. If multiple records have equal completeness, retain the first occurrence.
3. Store removed records in an audit table for traceability.

This ensures deterministic and reproducible data cleaning without fabricating information.

Completeness Score for passenger records

In [18]:
# Completeness score for every passenger record

passengers = dfs["passengers"].copy()

passengers["completeness_score"] = passengers.notna().sum(axis=1)

passenger_duplicates_score = (
    passengers[
        passengers["passenger_id"].duplicated(keep=False)
    ]
    .sort_values(["passenger_id","completeness_score"],
                 ascending=[True,False])
)

passenger_duplicates_score.head(20)

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth,completeness_score
34,P1034,Ishaan,Iyer,62,M,ishaan.iyer@outlook.com,+91-8679984033,122362316658,1964-08-14,9
35,P1034,Ishaan,Iyer,62,M,ishaan.iyer@gmail.com,+91-7994310083,669096705466,1964-09-24,9
103,P1102,Myra,Kulkarni,1,F,myra.kulkarni@rediffmail.com,+91-8128636900,606853615305,2025-02-27,9
104,P1102,Myra,Kulkarni,1,F,myra.kulkarni@outlook.com,+91-7247746438,47277901043,2025-03-18,9
106,P1104,Vivaan,Verma,88,F,vivaan.verma@gmail.com,+91-7871531100,246095396218,1938-06-03,9
...,...,...,...,...,...,...,...,...,...,...
224,P1216,Riya,Sharma,56,F,riya.sharma@outlook.com,+91-7367942460,154100676267,1970-07-12,9
225,P1216,Riya,Sharma,56,F,riya.sharma@hotmail.com,+91-9208908082,793271040541,1970-02-11,9
245,P1236,Myra,Sharma,20,F,myra.sharma@yahoo.com,+91-9821413281,584282417248,2006-03-22,9
246,P1236,MyraSharma,NaN,20,F,myrasharma.user@outlook.com,+91-6338642889,534655494717,2006-01-28,8


# Final Data Profiling Summary

## Objective

The objective of this notebook was to profile the raw ASG Airlines dataset before building the ETL pipeline. The analysis focused on understanding the dataset structure, identifying relationships between tables, measuring data quality, and defining business rules for cleaning.

---

## Dataset Overview

The workbook contains four relational tables:

| Table | Purpose |
|--------|---------|
| Flights | Flight schedule and operational data |
| Passengers | Passenger master information (PII) |
| Bookings | Booking transactions linking passengers and flights |
| Payments | Payment details for each booking |

The Bookings table acts as the central transactional table and connects the remaining entities through foreign keys.

---

## Key Findings

- 4 relational tables were successfully ingested.
- Total records analysed: **4,059**
- Missing values identified across all tables: **144**
- Duplicate rows detected: **15**
- Corrupted primary keys detected in Flights and Passengers.
- Personally Identifiable Information (PII) exists and will require masking before reporting.

---

## Cleaning Strategy (Business Rules)

| Issue | Strategy |
|--------|----------|
| Missing airline | Infer from flight ID prefix |
| Missing booking status | Replace with `UNKNOWN` |
| Missing passenger last name | Replace with `Unknown` |
| Missing payment amount | Preserve as NULL |
| Exact duplicate flight rows | Remove duplicates |
| Duplicate passenger IDs | Retain master record using completeness score |
| Conflicting flight IDs | Preserve using surrogate keys |

These rules will be implemented during the Silver Layer transformation in the ETL pipeline.


 FINAL EXECUTIVE DATA PROFILING REPORT


In [ ]:


total_rows = sum(df.shape[0] for df in dfs.values())
total_columns = sum(df.shape[1] for df in dfs.values())
total_missing = sum(df.isna().sum().sum() for df in dfs.values())
total_duplicates = sum(df.duplicated().sum() for df in dfs.values())

pk_duplicates = {
    "Flights": dfs["flights"]["flight_id"].duplicated().sum(),
    "Passengers": dfs["passengers"]["passenger_id"].duplicated().sum(),
    "Bookings": dfs["bookings"]["booking_id"].duplicated().sum(),
    "Payments": dfs["payments"]["payment_id"].duplicated().sum()
}

summary_df = pd.DataFrame({
    "Metric": [
        "Total Tables",
        "Total Records",
        "Total Columns",
        "Missing Values",
        "Duplicate Rows",
        "Flight PK Duplicates",
        "Passenger PK Duplicates"
    ],
    "Value": [
        len(dfs),
        total_rows,
        total_columns,
        total_missing,
        total_duplicates,
        pk_duplicates["Flights"],
        pk_duplicates["Passengers"]
    ]
})

summary_df

,Metric,Value
0,Total Tables,4
1,Total Records,4059
2,Total Columns,29
3,Missing Values,144
4,Duplicate Rows,15
5,Flight PK Duplicates,16
6,Passenger PK Duplicates,39


##  Conclusion

The raw dataset is not analytics-ready due to missing values, duplicate primary keys, inconsistent master records, and the presence of sensitive passenger information.

The profiling phase established deterministic business rules that will guide the Silver Layer transformation:

- Standardize and validate flight identifiers
- Resolve duplicate passenger master records
- Handle missing categorical values
- Preserve financial integrity by not imputing payment amounts
- Mask Aadhaar, Passport, Email, and Phone before reporting
